# Akshara Telugu OCR · controlled Colab continuation
Use **Runtime → Change runtime type → GPU**. Run cells in order. Inputs remain private in your Drive. This notebook reuses the existing IndicOCR trainer, not a replacement model.

Frozen protocol: **25,000 training pairs / 5,000 validation / 3,000 sealed final test; one initial epoch**. Source labels are not all human verified; writer IDs and possible pretraining overlap are unknown. The historical 80,083/two-epoch plan is retired. Final-test results never choose the model. No 99% or speedup claim is made.

Upload the contents of the supplied `Akshara_Colab_Transfer_v2` folder (the sibling directory of the project checkout) into **MyDrive/akshara-telugu-ocr/inputs/colab-transfer-v1/**, including `transfer-index.json` and every ZIP. The Drive folder name remains v1 for compatibility. If v1 is already uploaded, replace only `akshara-source.zip` and `transfer-index.json` with the v2 copies. Also upload this notebook to Colab. Keep prior runs and original files. A signed-in Hugging Face browser does not mount Drive or supply Colab compute. No Hugging Face credentials are required: the original authorized local checkpoint is transferred intact.

This workflow currently preserves BF16. A GPU without native BF16 support stops; FP16/GradScaler is not implemented. Do not convert an existing checkpoint to a different precision. An FP32 experiment would require a separate lineage and matched baselines.

## 1 · Environment and GPU availability (safe to rerun)

In [ ]:
import os,sys,json,shutil,subprocess,time,uuid
from pathlib import Path
print("Python:",sys.version,"CPU count:",os.cpu_count())
print("Local free disk GiB:",shutil.disk_usage("/content").free/2**30)
r=subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,memory.free,driver_version","--format=csv"],capture_output=True,text=True)
print(r.stdout or r.stderr)
if r.returncode: raise RuntimeError("No NVIDIA GPU available. Select a GPU runtime before continuing.")
LOCAL=Path("/content/akshara")
BOOTSTRAP=Path("/content/akshara-bootstrap")
VENV=Path("/content/akshara-venv")
PY=str(VENV/"bin/python")


## 2 · Private Drive mount and single-runtime ownership
Authorize the Google prompt yourself. Set the confirmation below only after verifying the Windows coordinator is held and no other Colab runtime is training this run. A stale lease is never automatically stolen. Drive filesystem locking alone is not a distributed concurrency guarantee.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE=Path("/content/drive/MyDrive/akshara-telugu-ocr")
INBOX=DRIVE/"inputs/colab-transfer-v1"
RUN_NAME="word-efficient-v3-colab-bf16"
DURABLE=DRIVE/"runs"/RUN_NAME
I_CONFIRM_NO_OTHER_RUNTIME_IS_TRAINING=False  # Change after checking the previous runtime.
RELEASE_PREVIOUS_OWNER=False  # Only after explicitly stopping/confirming a previous runtime is gone.
if not I_CONFIRM_NO_OTHER_RUNTIME_IS_TRAINING: raise RuntimeError("Confirm a single active training runtime before continuing.")
DRIVE.mkdir(parents=True,exist_ok=True)
LEASE=DRIVE/(RUN_NAME+".owner.json")
if "OWNER" not in globals(): OWNER=uuid.uuid4().hex
if LEASE.exists():
    previous=json.loads(LEASE.read_text())
    if previous["owner"]!=OWNER:
        if not RELEASE_PREVIOUS_OWNER: raise RuntimeError("Previous runtime lease remains. Verify it is stopped before releasing it.")
        LEASE.rename(DRIVE/(LEASE.name+".released-"+uuid.uuid4().hex))
if not LEASE.exists():
    with LEASE.open("x") as f: json.dump({"owner":OWNER,"created":time.time(),"run":RUN_NAME},f)
print("Private inputs:",INBOX,"Durable run snapshots:",DURABLE)


## 3 · Verify source archive and install compatible dependencies
Uses an isolated Python environment, leaving the Colab kernel packages alone. Exact Windows-tested versions are pinned; Linux installation and the GPU smoke test must still succeed. No optional kernels or compilation are enabled.

In [ ]:
import hashlib,zipfile
def file_sha(p):
    h=hashlib.sha256()
    with Path(p).open("rb") as f:
        for b in iter(lambda:f.read(8*1024*1024),b""): h.update(b)
    return h.hexdigest()
index=json.loads((INBOX/"transfer-index.json").read_text())
source=next(x for x in index["archives"] if x["file"]=="akshara-source.zip")
if file_sha(INBOX/source["file"])!=source["sha256"]: raise RuntimeError("Source archive checksum failed")
if not (BOOTSTRAP/"project/ocr/colab_storage.py").exists():
    with zipfile.ZipFile(INBOX/source["file"]) as z:
        for m in z.infolist():
            target=(BOOTSTRAP/m.filename).resolve()
            if not target.is_relative_to(BOOTSTRAP.resolve()) or "\\" in m.filename or ":" in m.filename or (m.external_attr>>16)&0o170000==0o120000: raise RuntimeError("Unsafe source ZIP")
        z.extractall(BOOTSTRAP)
if not Path(PY).exists(): subprocess.run([sys.executable,"-m","venv",str(VENV)],check=True)
subprocess.run([PY,"-m","pip","install","--disable-pip-version-check","torch==2.11.0","--index-url","https://download.pytorch.org/whl/cu128"],check=True)
subprocess.run([PY,"-m","pip","install","--disable-pip-version-check","-r",str(BOOTSTRAP/"project/requirements-colab.txt")],check=True)
subprocess.run([PY,"-m","pip","check"],check=True)
ENV=os.environ.copy(); ENV["PYTHONPATH"]=str(BOOTSTRAP/"project")


## 4 · Stage selected images and original model on local runtime disk
The archive set is copied/extracted once, not read from Drive for every training step. Checksums are verified. A partial stage is preserved and reported; choose a new empty directory or inspect it rather than deleting data blindly. Repeated successful staging verifies existing files.

In [ ]:
subprocess.run([PY,"-m","ocr.colab_storage","stage",str(INBOX),str(LOCAL)],env=ENV,check=True)
PROJECT=LOCAL/"project"
RUN=LOCAL/"runs"/RUN_NAME
ENV.update(PYTHONPATH=str(PROJECT),INDIC_OCR_REPO=str(LOCAL/"inputs/model"),OCR_DATA_ROOT=str(LOCAL/"inputs/data"),OCR_PILOT_ADAPTER=str(LOCAL/"inputs/pilot"),OCR_RUN_ROOT=str(RUN),OCR_PRECISION="bf16",OCR_COMPUTE_PROVIDER="google-colab",OCR_CHECKPOINT_EVERY_UPDATES="125")
ENV["OCR_PATH_MAP"]=json.dumps({"D:/OCR/models/indic-ocr":str(LOCAL/"inputs/model"),"D:/OCR/research-v2":str(LOCAL/"inputs/data"),"D:/OCR/runs/word-pilot-v1/trained/adapter":str(LOCAL/"inputs/pilot")})
print("Direct microbatch1 trainer; DataLoader worker/pin/persistent settings are not used.")


## 5 · Hardware, manifests, checkpoint recovery and software tests
Cloud validation starts a separate hardware lineage using identical frozen examples. The completed Windows baseline remains historical and is not mixed into a cloud prediction file. Existing cloud checkpoints resume; they do not restart from the base. Restore requires an empty local run directory.

In [ ]:
hardware_code="""import torch,psutil,json,shutil,os; assert torch.cuda.is_available(),'CUDA unavailable'; assert torch.cuda.is_bf16_supported(including_emulation=False),'Native BF16 GPU required'; p=torch.cuda.get_device_properties(0); print(json.dumps({'cuda_available':True,'gpu':p.name,'device_count':torch.cuda.device_count(),'capability':torch.cuda.get_device_capability(0),'total_vram':p.total_memory,'free_total_vram':torch.cuda.mem_get_info(),'torch':torch.__version__,'cuda':torch.version.cuda,'cpu_count':os.cpu_count(),'system_ram':psutil.virtual_memory()._asdict(),'disk_free':shutil.disk_usage('/content').free,'data_root':os.environ['OCR_DATA_ROOT'],'data_loader':'direct sequential microbatch1; no DataLoader workers'},indent=2))"""
hardware=subprocess.check_output([PY,"-c",hardware_code],env=ENV,text=True)
print(hardware)
if not RUN.exists() or not any(RUN.iterdir()):
    if DURABLE.exists() and any(DURABLE.glob("snapshot-*")):
        subprocess.run([PY,"-m","ocr.colab_storage","restore",str(DURABLE),str(RUN)],env=ENV,check=True)
    else:
        RUN.mkdir(parents=True,exist_ok=True)
        for name in ["protocol.json","train.jsonl","validation.jsonl","final-test.jsonl"]: shutil.copy2(LOCAL/"inputs/protocol"/name,RUN/name)
(RUN/("hardware-"+OWNER+".json")).write_text(hardware)
subprocess.run([PY,"-m","ocr.colab_storage","verify",str(LOCAL)],env=ENV,check=True)
subprocess.run([PY,"-m","unittest","discover","-s","tests","-q"],cwd=PROJECT,env=ENV,check=True)
subprocess.run([PY,"-m","pip","freeze"],stdout=(RUN/("environment-"+OWNER+".txt")).open("w"),check=True)
print((RUN/"status.json").read_text() if (RUN/"status.json").exists() else "Ready for original/pilot validation; no full candidate exists yet.")


## 6 · Actual GPU interrupted-versus-uninterrupted smoke
Safe to skip only if a verified passing result already exists in this cloud lineage with the same config and precision. The coordinator later verifies the validation-selected initialization separately. These eight-example smoke weights are never selected for deployment.

In [ ]:
SMOKE=RUN/"colab-preflight-smoke"
if not (SMOKE/"smoke_passed.json").exists():
    subprocess.run([PY,"-u","-m","ocr.scale_experiment","smoke","--config",str(PROJECT/"configs/colab-base.json"),"--manifest",str(RUN/"train.jsonl"),"--output",str(SMOKE)],cwd=PROJECT,env=ENV,check=True)
print((SMOKE/"smoke_passed.json").read_text())
subprocess.run([PY,"-m","ocr.colab_storage","snapshot",str(RUN),str(DURABLE)],env=ENV,check=True)


## 7 · Measure actual throughput before a long run
64 real training samples, eight warm-up samples excluded from steady-state timing. Full25k scheduler, rank8, accumulation4 and LR1e-5 remain unchanged. GPU timings synchronize; checkpoint writes and GPU-utilization samples are recorded. Benchmark weights stay separate. Compare against the supplied Windows benchmark only under identical config/precision/versions. A short projection excludes evaluation, Drive I/O and disconnects.

In [ ]:
BENCH=RUN/"colab-benchmark"
if not (BENCH/"benchmark.json").exists():
    if BENCH.exists() and any(BENCH.iterdir()): raise RuntimeError("Interrupted benchmark: choose a new benchmark output name; do not reuse partial timings.")
    subprocess.run([PY,"-u","-m","ocr.colab_benchmark","--config",str(PROJECT/"configs/colab-base.json"),"--manifest",str(RUN/"train.jsonl"),"--validation",str(RUN/"validation.jsonl"),"--output",str(BENCH),"--updates","16"],cwd=PROJECT,env=ENV,check=True)
benchmark=json.loads((BENCH/"benchmark.json").read_text())
print({k:v for k,v in benchmark.items() if k not in {"input_observations","gpu_samples","losses"}})
subprocess.run([PY,"-m","ocr.colab_storage","snapshot",str(RUN),str(DURABLE)],env=ENV,check=True)


## 8 · Resume the appropriate capped stage
Review the benchmark first. Set the launch flag when ready. The existing coordinator compares original and pilot on5k validation, selects initialization, runs exact-resume smoke, trains one25k epoch, selects by validation with paired intervals, locks selection, then evaluates original/pilot/candidate once on sealed3k test. No automatic extra epoch or revived80k plan. A completed study is not tested again. The runtime budget requests a safe stop; it does not guarantee Colab will stay connected that long.

In [ ]:
START_CONTROLLED_PIPELINE=False  # Set after reviewing actual benchmark and ensuring single ownership.
if not START_CONTROLLED_PIPELINE: raise RuntimeError("Review benchmark and enable the launch flag when ready.")
if (RUN/"STOP").exists(): raise RuntimeError("Previous safe-stop sentinel exists; use cell12 resume procedure first.")
if "trainer" in globals() and trainer.poll() is None: raise RuntimeError("This runtime already has an active trainer")
runner_log=(RUN/"colab-controller.log").open("a")
trainer=subprocess.Popen([PY,"-u","-m","ocr.colab_runner","--durable",str(DURABLE),"--seconds","10800","--sync-seconds","300"],cwd=PROJECT,env=ENV,stdout=runner_log,stderr=subprocess.STDOUT)
print("Controller PID:",trainer.pid,"Progress/logs:",RUN,"Rerun cell9 to inspect; do not launch twice.")


## 9 · Checkpoint and progress inspection (safe to rerun)
Snapshots occur about every five minutes. Checkpoints commit every125 optimizer updates and at epoch/stop boundaries. A sudden disconnect may lose work since the last verified Drive snapshot; previous snapshots remain available. Drive durability is verified by read-back hashes, not an absolute service guarantee.

In [ ]:
print("Controller return code:",trainer.poll() if "trainer" in globals() else "not owned by this kernel")
for name in ["status.json","drive-sync-status.json","full/training_progress.json","full/best.json","failure.json"]:
    p=RUN/name
    if p.exists(): print(name,p.read_text()[:4000])
for p in RUN.glob("experiments/*/progress.json"): print(p.parent.name,p.read_text())
print((RUN/"colab-controller.log").read_text()[-5000:] if (RUN/"colab-controller.log").exists() else "No controller log yet")


## 10 · Validation and final export
Final metrics appear only after validation selection is locked. A new adapter is not necessarily an improvement. Final chosen adapter may remain the original or pilot. API parity and page/sentence evaluation are separate remaining deployment checks.

In [ ]:
summary=RUN/"final-summary.json"
if summary.exists():
    result=json.loads(summary.read_text()); print("Locked selection:",result["selection"])
    for name,m in result["final_metrics"].items(): print(name,{k:m[k] for k in ["examples","micro_cer","micro_wer","raw_exact_match"]})
    export=DRIVE/"final-models"/RUN_NAME; export.mkdir(parents=True,exist_ok=True)
    package=PROJECT.parent/"Telugu_OCR_Capped_Selected_Model.zip"
    if package.exists():
        target=export/package.name
        if target.exists() and file_sha(target)!=file_sha(package): raise RuntimeError("Preserve existing final export; choose a versioned destination")
        shutil.copy2(package,target)
        if file_sha(target)!=file_sha(package): raise RuntimeError("Final export transfer failed")
    print("Final model folder:",export)
else: print("Training/evaluation unfinished. Do not claim final accuracy or deploy a partial adapter.")


## 11 · Explicit verified sync (safe to rerun)
Snapshots contain separate checkpoints, logs, validation results and—only after locked selection—final-test outputs. Older versioned snapshots are retained. This cell does not duplicate images/model files or start a GPU job.

In [ ]:
subprocess.run([PY,"-m","ocr.colab_storage","snapshot",str(RUN),str(DURABLE)],env=ENV,check=True)
print("Durable pointer:",(DURABLE/"latest-snapshot.json").read_text())


## 12 · Safe stop, release ownership, and resume after a disconnect
To stop: run the stop portion and keep the runtime open until the controller exits and sync succeeds. After disconnect: create a GPU runtime, repeat cells1–5, explicitly confirm the old runtime is gone/release its lease, restore latest intact snapshot, then rerun smoke/benchmark checks and launch cell8. Remove only the local STOP sentinel after the previous process has exited. Never delete checkpoints or reset the run to the original model to work around a failure.

In [ ]:
REQUEST_SAFE_STOP=True
if REQUEST_SAFE_STOP:
    (RUN/"STOP").write_text("User requested safe optimizer/prediction boundary stop.\n")
    if "trainer" in globals():
        try: trainer.wait(timeout=180)
        except subprocess.TimeoutExpired: raise RuntimeError("Still stopping safely; keep runtime open and rerun this cell. No process was killed.")
    else:
        # Verify no orphan controller/worker remains before releasing ownership.
        r=subprocess.run(["pgrep","-af","ocr.(colab_runner|run_efficient|scale_experiment)"],capture_output=True,text=True)
        if r.stdout.strip(): raise RuntimeError("A training process remains; inspect it before releasing ownership: "+r.stdout)
    subprocess.run([PY,"-m","ocr.colab_storage","snapshot",str(RUN),str(DURABLE)],env=ENV,check=True)
    if LEASE.exists() and json.loads(LEASE.read_text())["owner"]==OWNER: LEASE.rename(DRIVE/(LEASE.name+".released-"+OWNER))
    print("Safely stopped and synchronized. To resume, reacquire ownership in cell2; remove local STOP only after confirming no active worker.")
# Explicit resume after the checks above, in a later execution:
# (RUN/"STOP").unlink(missing_ok=True)
# rerun cell2 to acquire ownership, then cell8 with the same RUN_NAME/config/precision.


## Evidence and limitations
Read `COLAB_MIGRATION.md`, `PROJECT_STATE.md`, `WORK_LOG.md`, and the recorded local smoke/benchmark evidence. Notebook code cells have been parsed and structure-validated locally; Colab authentication, Linux package installation, cloud GPU resume, throughput and final training require actual execution.

Dataset attribution: IIIT Hyderabad, Gongidi & Jawahar, IIIT-INDIC-HW-WORDS (ICDAR2021), CC BY4.0, https://ilocr.iiit.ac.in/dataset/23/. Retain source attribution and license records. Original IndicOCR license files are transferred intact; verify their terms before redistributing weights. Keep all artifacts private.

[Colab resource and Drive limitations](https://research.google.com/colaboratory/faq.html): hardware availability and session duration vary. Local VM storage is temporary; a disconnect can lose unsynchronized work. This notebook does not bypass limits or guarantee a GPU/speedup.
